# Bonus B1 solution, notebook 2: give the notice a node of its own, and reject a request

The worked answers to the two challenge tasks of `b1b_durable_workflows_at_the_desk.ipynb`. The setup repeats
what the tasks need, so this file runs on its own, in one kernel, with no restart. It calls no model and needs
no key. Its tables and checkpoints go to `solved/outputs/`, apart from the notebook's.

## Setup

In [1]:
from contextlib import closing
from pathlib import Path
import sqlite3
import sys

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# The workflow's code is in a file in this lab's folder: let Python import from there
LAB = ROOT / "bonus" / "B1_LangGraph_Durable_Workflows"
sys.path.insert(0, str(LAB))

import refund_workflow
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)
for old_file in OUTPUTS.glob("*.db*"):
    old_file.unlink()
refund_workflow.create_desk_tables()

saver = SqliteSaver(sqlite3.connect(OUTPUTS / "checkpoints.db", check_same_thread=False))

refund_workflow.show_desk()

notices sent:    none
refund_requests: empty


## Task 1: Give the notice a node of its own

Two new nodes and one new route:

- `notify` sends the notice and finishes. It adds nothing to the state, so it returns `{}`.
- `review` holds only the `interrupt` and the decision. It has no line before the `interrupt`.
- `after_lookup` sends a found invoice to `notify`, where the file's route sends it to `review`.

The lookup, the filing and `after_review` come from `refund_workflow.py` unchanged.

In [2]:
def notify(state):
    refund_workflow.notify_reviewer(state)
    return {}


def review(state):
    decision = interrupt({field: state[field] for field in refund_workflow.PACKET_FIELDS})
    if decision == "approve":
        return {"decision": "approve"}
    return {"decision": "reject", "status": f"not filed: the reviewer said: {decision}"}


def after_lookup(state):
    if state.get("status"):
        return END
    return "notify"


builder = StateGraph(refund_workflow.Refund)

builder.add_node("look_up_invoice", refund_workflow.look_up_invoice)
builder.add_node("notify", notify)
builder.add_node("review", review)
builder.add_node("file_request", refund_workflow.file_request)

builder.add_edge(START, "look_up_invoice")
builder.add_conditional_edges("look_up_invoice", after_lookup, ["notify", END])
builder.add_edge("notify", "review")
builder.add_conditional_edges("review", refund_workflow.after_review, ["file_request", END])
builder.add_edge("file_request", END)

desk_v3 = builder.compile(checkpointer=saver)

print(list(desk_v3.get_graph().nodes))

['__start__', 'look_up_invoice', 'notify', 'review', 'file_request', '__end__']


Four nodes. Request `R-1003`: Puja Srivastava was charged twice for invoice 229, 13.86 US dollars.

In [3]:
request = {"request_id": "R-1003", "invoice_id": 229, "amount": 13.86, "reason": "charged twice"}
config = {"configurable": {"thread_id": request["request_id"]}}

paused = desk_v3.invoke(request, config)

print("for the reviewer:", paused["__interrupt__"][0].value)
print("next node:       ", desk_v3.get_state(config).next)

LOOKED UP invoice 229
NOTICE SENT to the reviewer for R-1003
for the reviewer: {'request_id': 'R-1003', 'customer': 'Puja Srivastava', 'invoice_id': 229, 'invoice_total': 13.86, 'amount': 13.86, 'reason': 'charged twice'}
next node:        ('review',)


The lookup found Puja Srivastava and a total of 13.86, the notice went out, and the run waits at `review`.
`notify` has finished, so it has a checkpoint. We approve.

In [4]:
done = desk_v3.invoke(Command(resume="approve"), config)

print("status:", done["status"])
refund_workflow.show_desk()

REFUND REQUEST FILED for R-1003
status: filed
notices sent:    ['R-1003']
refund_requests: [('R-1003', 229, 13.86)]


The approval printed only `REFUND REQUEST FILED for R-1003`. There is no second `NOTICE SENT` line, and `notices`
holds one row for `R-1003`. `review` did start again from its first line, and its first line is the `interrupt`.
The notice is a plain insert with no check: it ran once because its node had finished before the run stopped.

## Task 2: Reject a request

The task uses the notebook's `desk_v2`: the file's workflow with `review_once`, the review node that asks the
`notices` table before sending. We build it again here.

In [5]:
def notify_reviewer_once(state):
    with closing(sqlite3.connect(refund_workflow.DESK_DB)) as desk_db:
        sent = desk_db.execute("SELECT COUNT(*) FROM notices WHERE request_id = ?",
                               (state["request_id"],)).fetchone()[0]
    if sent:
        print("NOTICE ALREADY SENT for", state["request_id"], "(not sent again)")
    else:
        refund_workflow.notify_reviewer(state)


def review_once(state):
    notify_reviewer_once(state)
    decision = interrupt({field: state[field] for field in refund_workflow.PACKET_FIELDS})
    if decision == "approve":
        return {"decision": "approve"}
    return {"decision": "reject", "status": f"not filed: the reviewer said: {decision}"}


desk_v2 = refund_workflow.build_graph(saver, review_node=review_once)

Request `R-1004`: Helena Holý, invoice 175, 1.98 US dollars, charged twice.

In [6]:
request = {"request_id": "R-1004", "invoice_id": 175, "amount": 1.98, "reason": "charged twice"}
config = {"configurable": {"thread_id": request["request_id"]}}

paused = desk_v2.invoke(request, config)

print("for the reviewer:", paused["__interrupt__"][0].value)

LOOKED UP invoice 175
NOTICE SENT to the reviewer for R-1004
for the reviewer: {'request_id': 'R-1004', 'customer': 'Helena Holý', 'invoice_id': 175, 'invoice_total': 1.98, 'amount': 1.98, 'reason': 'charged twice'}


The reviewer checks billing and finds one charge. The decision is a reason in place of the word `approve`.

In [7]:
done = desk_v2.invoke(Command(resume="billing shows one charge for invoice 175"), config)

print("status:   ", done["status"])
print("next node:", desk_v2.get_state(config).next)
refund_workflow.show_desk()

NOTICE ALREADY SENT for R-1004 (not sent again)
status:    not filed: the reviewer said: billing shows one charge for invoice 175
next node: ()
notices sent:    ['R-1003', 'R-1004']
refund_requests: [('R-1003', 229, 13.86)]


No `REFUND REQUEST FILED` line. `review_once` returned the decision `reject` with the reason in `status`, and
`after_review` sent the run to the end: `file_request` never ran. `refund_requests` still holds only `R-1003`
from Task 1. `notices` has one row for `R-1004`: the resume found it and sent nothing more. `next` is empty,
so the thread is finished and nothing is waiting on it.

The rejection used the same two lines of code as an approval. What the reviewer's text means is decided by our
own `if` in the review node and by `after_review`.

Back to [the notebook](../b1b_durable_workflows_at_the_desk.ipynb).